In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 4
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.277908293844096

Trial 1 =========================================
14.246343086344908

Trial 2 =========================================
14.452077555939747

Trial 3 =========================================
17.569197853347003

Trial 4 =========================================
15.394193423028678

Trial 5 =========================================
15.397863657737469

Trial 6 =========================================
17.54991908348002

Trial 7 =========================================
14.464787905243892

Trial 8 =========================================
14.45717141272066

Trial 9 =========================================
16.17862018085033

Trial 10 =========================================
17.52399318369555

Trial 11 =========================================
14.028079941321252

Trial 12 =========================================
14.809051954778202

Trial 13 =========================================
15.40043586321283

Trial 14 ============

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 18 =========================================
16.047017004577583

Trial 19 =========================================
15.29519652921699

Trial 20 =========================================
15.35142001178001

Trial 21 =========================================
17.559236587927046

Trial 22 =========================================
14.63548177996702

Trial 23 =========================================
14.289530410513915

Trial 24 =========================================
13.7711127612637

Trial 25 =========================================
15.387486600434897

Trial 26 =========================================
14.605495625624284

Trial 27 =========================================
16.237142351997946



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 28 =========================================
16.035655698203165

Trial 29 =========================================
16.24325215353178



...

Trial 58 =========================================
15.933968579660034

Trial 59 =========================================
16.21825511171639

Trial 60 =========================================
15.879245740118272

Trial 61 =========================================
15.388680521668359

Trial 62 =========================================
13.842629869113445

Trial 63 =========================================
14.638278356512263

Trial 64 =========================================
18.753233725576052

Trial 65 =========================================
15.387745275941246

Trial 66 =========================================
15.552638761448147

Trial 67 =========================================
15.395177934159076

Trial 68 =========================================
15.302149392294053

Trial 69 =========================================
15.389742213981016

Trial 70 =========================================
14.90129843220828

Trial 71 =========================================
14.265555492499942

Trial 72

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 80 =========================================
16.132074595482447

Trial 81 =========================================
14.466219270615976

Trial 82 =========================================
16.174072366301846

Trial 83 =========================================
14.459175670780805

Trial 84 =========================================
15.380162388035602

Trial 85 =========================================
15.263011385373812

Trial 86 =========================================
14.67841781218059

Trial 87 =========================================
18.8018943358007



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.13/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 88 =========================================
16.228581408406228

Trial 89 =========================================
16.345795184738005

Trial 90 =========================================
15.372620971560034

Trial 91 =========================================
15.39646189598417

Trial 92 =========================================
15.972695223207001

Trial 93 =========================================
12.862767563589463

Trial 94 =========================================
15.455904133788442

Trial 95 =========================================
14.374058431653129

Trial 96 =========================================
14.495524679511433

Trial 97 =========================================
15.395769369254532

Trial 98 =========================================
18.798500207163666

Trial 99 =========================================
15.420433656565073

[14.27790829 14.24634309 14.45207756 17.56919785 15.39419342 15.39786366
 17.54991908 14.46478791 14.45717141 16.17862018 17.52399318 14.02807994
 14

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.818897126432823
Avg = 15.581773093706893
Std = 1.3745366201611615


In [6]:
print(y_max_arr.tolist())

[14.277908293844096, 14.246343086344908, 14.452077555939747, 17.569197853347003, 15.394193423028678, 15.397863657737469, 17.54991908348002, 14.464787905243892, 14.45717141272066, 16.17862018085033, 17.52399318369555, 14.028079941321252, 14.809051954778202, 15.40043586321283, 16.18491008872226, 15.393106496939925, 14.274304534069831, 18.17241378655847, 16.047017004577583, 15.29519652921699, 15.35142001178001, 17.559236587927046, 14.63548177996702, 14.289530410513915, 13.7711127612637, 15.387486600434897, 14.605495625624284, 16.237142351997946, 16.035655698203165, 16.24325215353178, 15.962438093386677, 15.396462444241568, 18.75722127755609, 14.271114049221644, 13.771653321353348, 14.629805579756894, 14.437533880281256, 14.83492371106376, 15.392258290528831, 18.793860576725315, 14.483734021442615, 14.167869258172443, 14.009530162478212, 14.453164306318138, 18.818897126432823, 17.29158087547072, 14.280162384145815, 13.942158840322715, 17.5829518422232, 14.161256380131801, 18.78080149591024

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-4.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)